# Measurement on a Google Colab T4

Required part of Phase M: runs `qcclass medir` on a Colab **Tesla T4** (16 GB, plain and
with mixed precision, both with the 25k–400k node-budget sweep) and copies every result JSON
back to Drive.

Prepare once, on the machine that holds the dataset:

```bash
# data.tgz: only what the measurement reads (VERSION.json comes with the repository)
COPYFILE_DISABLE=1 tar czf data.tgz data/metadata.csv data/splits data/graphs/dag data/graphs/etiquetas.json
```

Upload `data.tgz` to `MyDrive/QTagger/`. If the repository is private, also upload an archive
of it (`git archive --format=tar.gz -o qtagger.tgz HEAD`) and set `REPO_ARCHIVE` below.

Runtime: *Runtime → Change runtime type → T4 GPU*. The pinned torch 2.15.0 is built for
CUDA 13.2 and needs an NVIDIA driver 580 or newer. If Colab's driver is older, or the build
cannot run on the T4, the notebook falls back to torch 2.14.1 + CUDA 12.6; the torch version
actually used is recorded in the JSON (`equipo.torch`, `equipo.cuda`).

In [ ]:
# --- settings ---
DRIVE_DIR = "/content/drive/MyDrive/QTagger"
REPO_URL = "https://github.com/SamRo28/QTagger.git"
REPO_ARCHIVE = None  # e.g. f"{DRIVE_DIR}/qtagger.tgz" for a private repository
GIT_REF = "main"
MACHINE = "colab-t4"  # results/medicion/<MACHINE>_cuda[_amp].json
WORKDIR = "/content/QTagger"
VENV = "/content/venv"
FALLBACK_TORCH = "torch==2.14.1+cu126"
FALLBACK_INDEX = "https://download.pytorch.org/whl/cu126"

In [ ]:
import subprocess

from google.colab import drive

drive.mount("/content/drive")
gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
).stdout.strip()
print(gpu or "no GPU")
if "T4" not in gpu:
    raise RuntimeError("this measurement needs a T4 runtime: Runtime -> Change runtime type -> T4 GPU")

In [ ]:
import os

if REPO_ARCHIVE:
    os.makedirs(WORKDIR, exist_ok=True)
    !tar xzf "{REPO_ARCHIVE}" -C "{WORKDIR}"
else:
    !git clone --depth 1 --branch "{GIT_REF}" "{REPO_URL}" "{WORKDIR}"
!tar xzf "{DRIVE_DIR}/data.tgz" -C "{WORKDIR}"
%cd {WORKDIR}

The package needs Python 3.13, which Colab may not provide: `uv` creates a 3.13
environment and installs the package pinned to `requirements.lock`.

In [ ]:
!pip install -q uv
!uv venv --python 3.13 "{VENV}"
!uv pip install --python "{VENV}/bin/python" -c requirements.lock -e .

In [ ]:
CHECK = (
    "import torch; x = torch.randn(256, 256, device='cuda'); (x @ x).sum().item(); "
    "print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))"
)


def cuda_works() -> bool:
    result = subprocess.run([f"{VENV}/bin/python", "-c", CHECK], capture_output=True, text=True)
    print(result.stdout.strip() or result.stderr.strip().splitlines()[-1])
    return result.returncode == 0


if not cuda_works():
    print(f"pinned torch cannot use this GPU/driver: installing {FALLBACK_TORCH}")
    !uv pip install --python "{VENV}/bin/python" --index-url "{FALLBACK_INDEX}" "{FALLBACK_TORCH}"
    if not cuda_works():
        raise RuntimeError("CUDA is not usable even with the fallback torch")

In [ ]:
def save_results() -> None:
    !mkdir -p "{DRIVE_DIR}/medicion"
    !cp results/medicion/*.json "{DRIVE_DIR}/medicion/"


!{VENV}/bin/qcclass verificar
!{VENV}/bin/qcclass medir --dispositivo cuda --max-lotes 200 --barrido --equipo "{MACHINE}"
save_results()

In [ ]:
!{VENV}/bin/qcclass medir --dispositivo cuda --max-lotes 200 --barrido --amp --equipo "{MACHINE}"
save_results()
!ls -l "{DRIVE_DIR}/medicion"

Copy the JSON files from `MyDrive/QTagger/medicion/` into `results/medicion/` of the
repository and run `qcclass medir --resumen`.